# Regresión Logística Binaria - Base Resumen

In [ ]:
#Cargamos librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
#Cargar archivo csv
df= pd.read_csv("base_resumen_limpio.csv")
df.head()

In [ ]:
#Rellenamos nulos
df = df.bfill()
df = df.ffill()

**Reemplazar valores**

In [ ]:
#Verificar los valores sin repetirse de una columna
unico = np.unique(df['Año'])
unico

In [ ]:
#Convertimos una variable categórica a dicotómica
#2024 se queda como "2024" y los años 2025 y 2026 se agrupan como "2025-2026"
df['Periodo']= df['Año'].astype(str).replace(["2025", "2026"], "2025-2026")
df[['Año', 'Mes', 'Periodo']]

In [ ]:
#Convertimos la variable dicotómica a número para usarla en el modelo
#1 = 2025-2026, 0 = 2024
df['Periodo_num']= np.where(df['Periodo'] == "2025-2026", 1, 0)
df.groupby(['Periodo'])['Periodo'].count()

**Creación de Categorias a partir de clases**

In [ ]:
#Obtenemos el limite superior y el límite inferior de la columna objetivo
Max=df['Ventas Real'].max()
Min=df['Ventas Real'].min()
Limites= [Min, Max]
Limites

In [ ]:
#Categorización de variables
#Declaramos 2 intervalos
intervalos=np.linspace(5.9, 42.1, 3)
intervalos

In [ ]:
#Creamos las categorías
categorias= ["Ventas bajas", "Ventas altas"]

In [ ]:
# Ajustar maximo de filas
pd.options.display.max_rows = None

In [ ]:
#Finalmente creamos las categorías en la columna numérica
df['Ventas_Cat']=pd.cut(x= df['Ventas Real'], bins=intervalos, labels= categorias )

In [ ]:
df[['Año', 'Mes', 'Ventas Real', 'Ventas_Cat']]

In [ ]:
#Analizar categorias de la variable dependiente
Categorias = df.groupby(['Ventas_Cat'], observed=True)['Ventas_Cat'].count()
Categorias

# **Modelo 1: Regresión logística con varias variables**

In [ ]:
#Declaramos las variables dependientes e independientes para la regresión Logística
Vars_Indep= df[['Leads', 'Efectivos', 'PDM', 'Periodo_num']]
Var_Dep= df['Ventas_Cat']

In [ ]:
#Redefinimos las variables
X= Vars_Indep
y= Var_Dep

In [ ]:
#Dividimos el conjunto de datos en la parte de entrenamiento y prueba:
#stratify mantiene la proporción de Ventas bajas y Ventas altas en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
#Se escalan todos los datos
escalar = StandardScaler()

In [ ]:
#Para realizar el escalamiento de las variables “X” tanto de entrenamiento como de prueba, utilizaremos fit_transform
X_train = escalar.fit_transform(X_train)
X_test = escalar.transform(X_test)

In [ ]:
#Definimos el algoritmo a utilizar
from sklearn.linear_model import LogisticRegression
algoritmo = LogisticRegression()

In [ ]:
#Entrenamos el modelo
algoritmo.fit(X_train, y_train)

In [ ]:
#Coeficientes del modelo
#Coeficiente positivo = aumenta la probabilidad de "Ventas bajas"
#(sklearn toma como clase 1 la segunda en orden alfabético: "Ventas bajas")
Coeficientes = pd.DataFrame({'Variable': Vars_Indep.columns,
                             'Coeficiente': algoritmo.coef_[0]}).round(3)
Coeficientes

In [ ]:
#Realizamos una predicción
y_pred = algoritmo.predict(X_test)
y_pred

In [ ]:
#Verifico la matriz de Confusión
from sklearn.metrics import confusion_matrix
matriz = confusion_matrix(y_test, y_pred)
print('Matriz de Confusión:')
print(matriz)

In [ ]:
#Calculo la precisión del modelo
from sklearn.metrics import precision_score

precision = precision_score(y_test, y_pred, average="binary", pos_label="Ventas bajas", zero_division=0)
print('Precisión del modelo:')
print(precision)

In [ ]:
#Calculo la exactitud del modelo
from sklearn.metrics import accuracy_score

exactitud = accuracy_score(y_test, y_pred)
print('Exactitud del modelo:')
print(exactitud)

In [ ]:
#Calculo la sensibilidad del modelo
from sklearn.metrics import recall_score

sensibilidad = recall_score(y_test, y_pred, average="binary", pos_label="Ventas bajas")
print('Sensibilidad del modelo:')
print(sensibilidad)

In [ ]:
# el puntaje F1 que es una combinación entre la precisión y la sensibilidad, para esto importamos f1_score.
#Calculo el Puntaje F1 del modelo
from sklearn.metrics import f1_score

puntajef1 = f1_score(y_test, y_pred, average="binary", pos_label="Ventas bajas")
print('Puntaje F1 del modelo:')
print(puntajef1)

# **Modelo 2: Regresión logística con una variable (Leads) y balanceo**

In [ ]:
#Declaramos las variables dependientes e independientes para la regresión Logística
Vars_Indep= df[['Leads']]
Var_Dep= df['Ventas_Cat']

In [ ]:
#Redefinimos las variables
X= Vars_Indep
y= Var_Dep

In [ ]:
#Dividimos el conjunto de datos en la parte de entrenamiento y prueba:
#stratify mantiene la proporción de Ventas bajas y Ventas altas en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
#Se escalan todos los datos
escalar = StandardScaler()

In [ ]:
#Para realizar el escalamiento de las variables “X” tanto de entrenamiento como de prueba, utilizaremos fit_transform
X_train = escalar.fit_transform(X_train)
X_test = escalar.transform(X_test)

In [ ]:
#Definimos el algoritmo a utilizar
algoritmo_Leads = LogisticRegression()

In [ ]:
#Entrenamos el modelo
algoritmo_Leads.fit(X_train, y_train)

In [ ]:
#Realizamos una predicción
y_pred_leads = algoritmo_Leads.predict(X_test)
y_pred_leads

In [ ]:
#Verifico la matriz de Confusión
from sklearn.metrics import confusion_matrix
matriz = confusion_matrix(y_test, y_pred_leads)
print('Matriz de Confusión:')
print(matriz)

In [ ]:
#Calculo la precisión del modelo
from sklearn.metrics import precision_score

precision = precision_score(y_test, y_pred_leads, average="binary", pos_label="Ventas bajas", zero_division=0)
print('Precisión del modelo:')
print(precision)

In [ ]:
#Calculo la exactitud del modelo
from sklearn.metrics import accuracy_score

exactitud = accuracy_score(y_test, y_pred_leads)
print('Exactitud del modelo:')
print(exactitud)

In [ ]:
#Calculo la sensibilidad del modelo
from sklearn.metrics import recall_score

sensibilidad = recall_score(y_test, y_pred_leads, average="binary", pos_label="Ventas bajas")
print('Sensibilidad del modelo:')
print(sensibilidad)

In [ ]:
# el puntaje F1 que es una combinación entre la precisión y la sensibilidad, para esto importamos f1_score.
#Calculo el Puntaje F1 del modelo
from sklearn.metrics import f1_score

puntajef1 = f1_score(y_test, y_pred_leads, average="binary", pos_label="Ventas bajas")
print('Puntaje F1 del modelo:')
print(puntajef1)

***Método 1 Reponderación de las Clases***

In [ ]:
#Reponderación de las clases
algoritmo_Pond = LogisticRegression(class_weight='balanced')
algoritmo_Pond.fit(X_train, y_train)

In [ ]:
#Realizamos una predicción
y_pred_pond = algoritmo_Pond.predict(X_test)
y_pred_pond

In [ ]:
#Verifico la matriz de Confusión
from sklearn.metrics import confusion_matrix
matriz = confusion_matrix(y_test, y_pred_pond)
print('Matriz de Confusión:')
print(matriz)

In [ ]:
#Calculo la precisión del modelo
from sklearn.metrics import precision_score

precision = precision_score(y_test, y_pred_pond, average="binary", pos_label="Ventas bajas", zero_division=0)
print('Precisión del modelo:')
print(precision)

In [ ]:
#Calculo la exactitud del modelo
from sklearn.metrics import accuracy_score

exactitud = accuracy_score(y_test, y_pred_pond)
print('Exactitud del modelo:')
print(exactitud)

In [ ]:
#Calculo la sensibilidad del modelo
from sklearn.metrics import recall_score

sensibilidad = recall_score(y_test, y_pred_pond, average="binary", pos_label="Ventas bajas")
print('Sensibilidad del modelo:')
print(sensibilidad)

In [ ]:
# el puntaje F1 que es una combinación entre la precisión y la sensibilidad, para esto importamos f1_score.
#Calculo el Puntaje F1 del modelo
from sklearn.metrics import f1_score

puntajef1 = f1_score(y_test, y_pred_pond, average="binary", pos_label="Ventas bajas")
print('Puntaje F1 del modelo:')
print(puntajef1)

***Método 2 Submuestreo o Sobremuestreo***

In [ ]:
%pip install imblearn

In [ ]:
#Aplicamos la técnica de sobremuestreo (oversampling)
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
#Revisamos cuántos registros quedan de cada clase
pd.Series(y_resampled).value_counts()

In [ ]:
#Entrenamos el modelo sobremuestreado
algoritmo_Over_sampling = LogisticRegression()
algoritmo_Over_sampling.fit(X_resampled, y_resampled)

In [ ]:
#Realizamos una predicción
y_pred_over_sampling = algoritmo_Over_sampling.predict(X_test)
y_pred_over_sampling

In [ ]:
#Verifico la matriz de Confusión
from sklearn.metrics import confusion_matrix
matriz = confusion_matrix(y_test, y_pred_over_sampling)
print('Matriz de Confusión:')
print(matriz)

In [ ]:
#Calculo la precisión del modelo
from sklearn.metrics import precision_score

precision = precision_score(y_test, y_pred_over_sampling, average="binary", pos_label="Ventas bajas", zero_division=0)
print('Precisión del modelo:')
print(precision)

In [ ]:
#Calculo la exactitud del modelo
from sklearn.metrics import accuracy_score

exactitud = accuracy_score(y_test, y_pred_over_sampling)
print('Exactitud del modelo:')
print(exactitud)

In [ ]:
#Calculo la sensibilidad del modelo
from sklearn.metrics import recall_score

sensibilidad = recall_score(y_test, y_pred_over_sampling, average="binary", pos_label="Ventas bajas")
print('Sensibilidad del modelo:')
print(sensibilidad)

In [ ]:
# el puntaje F1 que es una combinación entre la precisión y la sensibilidad, para esto importamos f1_score.
#Calculo el Puntaje F1 del modelo
from sklearn.metrics import f1_score

puntajef1 = f1_score(y_test, y_pred_over_sampling, average="binary", pos_label="Ventas bajas")
print('Puntaje F1 del modelo:')
print(puntajef1)

# **Comparativa de modelos**

In [ ]:
#Tabla comparativa de métricas de los modelos
etiqueta = "Ventas bajas"
predicciones = [y_pred_leads, y_pred_pond, y_pred_over_sampling]

tabla_metricas = pd.DataFrame({
    'Modelo': ['Leads sin balanceo', 'Leads + Reponderación', 'Leads + Oversampling'],
    'Exactitud': [accuracy_score(y_test, p) for p in predicciones],
    'Precisión': [precision_score(y_test, p, pos_label=etiqueta, zero_division=0) for p in predicciones],
    'Sensibilidad': [recall_score(y_test, p, pos_label=etiqueta) for p in predicciones],
    'Puntaje F1': [f1_score(y_test, p, pos_label=etiqueta) for p in predicciones]
}).round(3)

tabla_metricas